# B300 한 장에서 Qwen 추론 속도 측정하기

회의록 한 건을 요약할 때는 **얼마나 빨리 답이 나오는지**가 중요합니다. 여러 사람이 동시에 회의록을 보낸다면 **같은 시간에 얼마나 많이 처리하는지**도 알아야 합니다. 이 노트북에서는 두 값을 따로 잽니다.

| 순서 | 실행할 내용 | 비교할 값 |
| --- | --- | --- |
| 2절 | 같은 Qwen 모델과 같은 입력 8개를 Transformers와 vLLM으로 실행합니다. | 1,024토큰을 생성하는 데 걸린 시간 |
| 3절 | vLLM 서버를 켜고 회의 시간에 대한 질문을 보냅니다. | 모델 이름과 답변 내용 |
| 5~7절 | 동시 요청 수, 입력 길이, 출력 길이를 하나씩 바꿉니다. | 첫 응답 시간, 전체 응답 시간, 초당 출력 토큰 수 |

직접 실행과 서버 실험 모두 B300 한 장을 사용합니다. 두 Python 파일과 이 노트북을 같은 폴더에 두고 JupyterLab에서 엽니다. Qwen 모델을 지원하는 vLLM과 Transformers가 설치된 Python 커널을 선택합니다. [환경 준비](README.md) · [발표의 부하 테스트](chapter-04.html#workload)

## 1. 모델과 GPU 설정

`MODEL`은 `Qwen/Qwen3.8-27B`입니다. 기본 이름을 쓰면 필요한 모델 파일을 내려받습니다. 이미 있다면 로컬 폴더의 경로를 지정합니다. 가중치는 모델이 학습한 숫자입니다. BF16은 숫자 하나에 2바이트를 쓰므로, 270억 개라면 가중치만 약 54 GB입니다. 실제 실행에는 중간 계산값을 위한 메모리도 필요합니다.

`SINGLE_GPU`는 모델을 직접 실행할 GPU 번호, `모델 가중치와 요청의 중간 계산값을 한 GPU에 보관합니다. 다른 GPU로 결과를 보낼 필요가 없고, 사용 가능한 메모리는 한 장의 용량으로 제한됩니다._GPUS`는 서버가 쓸 GPU 번호입니다. `TENSOR_PARALLEL_SIZE`는 한 모델의 계산을 몇 장에 나눌지 정합니다. 장비에서 할당받은 GPU 번호를 사용합니다. 서버 실험은 한 조건당 256건을 보내고 같은 조건에서 세 번 반복합니다. [발표의 모델 파일](chapter-04.html#files)

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from urllib.request import Request, urlopen

MODEL = "Qwen/Qwen3.8-27B"
SINGLE_GPU = "0"
SERVER_GPUS = "0"
TENSOR_PARALLEL_SIZE = 1
BASE_URL = "http://127.0.0.1:8000"
NUM_PROMPTS = 256
REPEATS = 3

VLLM = [sys.executable, "-m", "vllm.entrypoints.cli.main"]

## 2. Transformers와 vLLM으로 같은 입력 계산하기

| 실행 파일 | 모델을 읽는 코드 | 측정할 작업 |
| --- | --- | --- |
| [transformers_baseline.py](transformers_baseline.py) | `from_pretrained(...)` | 입력 텐서를 `model.generate()`에 전달합니다. |
| [vllm_baseline.py](vllm_baseline.py) | `LLM(...)` | 같은 입력 토큰 번호를 vLLM의 `generate()`에 전달합니다. |

입력 길이는 512토큰, 출력 길이는 128토큰입니다. 8건을 함께 실행하므로 한 번에 8 × 128 = **1,024토큰**을 생성합니다. 같은 문장을 반복해 입력 길이를 맞추고, 출력 길이도 두 프로그램에서 똑같이 제한합니다.

**GPU 한 장에서 차례로** 실행합니다. 각 프로그램은 두 번 미리 계산한 후 세 번 시간을 잽니다. PyTorch의 `torch.cuda.synchronize()`는 요청한 GPU 계산이 끝날 때까지 기다립니다. 모델을 읽는 시간은 측정에서 제외합니다. Transformers는 전체 모델을 읽고, vLLM은 텍스트 부분만 읽습니다. [발표의 PyTorch 예제](chapter-04.html#pytorch)

In [ ]:
single_gpu_env = os.environ.copy()
single_gpu_env["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU

result = subprocess.run(
    [sys.executable, "transformers_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("Transformers: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

In [ ]:
result = subprocess.run(
    [sys.executable, "vllm_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("vLLM: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

### 결과 한 줄 읽기

출력 순서는 **경과 시간(초), 출력 토큰 수, 초당 출력 토큰 수**입니다. 예를 들어 1,024토큰을 4초에 생성했다면 256토큰/초입니다. 이 수치는 계산 예시입니다.

두 프로그램 모두 출력이 1,024토큰인지 먼저 확인하고 세 번의 경과 시간을 비교합니다. 여기서 측정한 값은 입력 8개를 모두 처리한 시간입니다. 어느 커널이 오래 걸렸는지 알아보려면 연산별 시간을 따로 재야 합니다. [발표의 프로파일링](chapter-04.html#profile)

## 3. vLLM 서버 실행하기

1. 아래 셀을 실행해서 서버 실행 명령을 확인합니다.
2. 같은 컴퓨터의 **별도 터미널**에서 출력된 한 줄을 실행합니다.
3. `Application startup complete`가 표시되면 서버를 켜 둔 채 노트북으로 돌아옵니다.

모델 가중치와 요청의 중간 계산값을 한 GPU에 보관합니다. 다른 GPU로 결과를 보낼 필요가 없고, 사용 가능한 메모리는 한 장의 용량으로 제한됩니다. 노트북은 `http://127.0.0.1:8000`으로 질문을 보냅니다.

| 설정 | 이 실습에서의 의미 |
| --- | --- |
| 모델 길이 4,096 | 한 요청에서 입력과 출력 토큰 수를 합한 상한입니다. |
| 최대 요청 수 32 | 엔진이 한 번에 함께 계산할 요청 수의 상한입니다. |
| 한 번에 계산할 토큰 4,096 | 그 단계에서 모든 요청을 합해 계산할 토큰 수의 상한입니다. |

반복된 입력도 매번 계산하도록 접두사 캐싱을 끕니다. 설치한 버전에서 `--disable-prefix-caching`을 지원하지 않으면 `vllm serve --help`에 나온 `--no-enable-prefix-caching`을 사용합니다. [발표의 연속 배칭](chapter-04.html#batch) · [Dynamo의 워커 배정](chapter-04.html#routing)

In [ ]:
server_command = [
    "env", f"CUDA_VISIBLE_DEVICES={SERVER_GPUS}",
    *VLLM, "serve", MODEL,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    "--dtype", "bfloat16",
    "--max-model-len", "4096",
    "--max-num-seqs", "32",
    "--max-num-batched-tokens", "4096",
    "--disable-prefix-caching",
    "--generation-config", "vllm",
    "--language-model-only",
]
print(shlex.join(server_command))

### 회의 공지로 서버 확인하기

`/v1/models`에서 서버의 모델 이름을 읽습니다. 이어서 `/v1/chat/completions`로 공지와 질문을 보냅니다. 공지에 따르면 변경된 시간은 **목요일 오후 4시**입니다. 모델 이름과 답이 맞는지 확인합니다. 이 요청은 뒤의 성능 측정에서 제외합니다.

In [ ]:
with urlopen(f"{BASE_URL}/v1/models") as response:
    models = json.load(response)
print([item["id"] for item in models["data"]])

request = Request(
    f"{BASE_URL}/v1/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": "목요일 회의를 오후 2시에서 4시로 변경합니다. 장소는 3층 소회의실입니다. 변경된 회의 시간이 언제인가요?",
        }],
        "temperature": 0,
        "chat_template_kwargs": {"enable_thinking": False},
        "max_tokens": 128,
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urlopen(request) as response:
    answer = json.load(response)
print(answer["choices"][0]["message"]["content"])

## 4. 부하 테스트 결과표 읽기

| 출력에 표시되는 이름 | 뜻 |
| --- | --- |
| Successful / Failed requests | 성공 256건, 실패 0건인지 먼저 확인합니다. |
| Output token throughput | 모든 요청의 출력 토큰 수를 전체 측정 시간(초)으로 나눈 값입니다. |
| TTFT | 요청을 보낸 후 첫 출력이 도착하기까지의 시간입니다. 서버 안에서 기다린 시간과 전송 시간도 포함합니다. |
| TPOT | 첫 출력 후 남은 시간을 남은 출력 토큰 수로 나눈 값입니다. |
| E2EL | 요청 전송부터 전체 답변을 받을 때까지의 시간입니다. |
| p50 / p95 | p50은 가운데 값입니다. 전체 응답 시간 p95가 800 ms라면 약 95%가 800 ms 이내에 끝났다는 뜻입니다. |

첫 토큰이 120 ms, 전체 답변이 180 ms에 도착하고 출력이 4토큰이라면 TPOT는 (180 − 120) ÷ 3 = **20 ms**입니다. [발표의 시간 계산 예](chapter-04.html#metrics)

아래 함수는 `vllm bench serve`로 길이를 맞춘 입력을 보냅니다. 8건을 미리 실행하고 본 측정에서 256건을 보냅니다. 동시 요청 상한이 4이면 네 건까지 보내고, 하나가 끝나는 대로 다음 요청을 보냅니다. 출력 길이를 고정하려고 종료 토큰이 나와도 정한 길이까지 생성합니다. 이 실험으로는 답변 품질을 알 수 없습니다. 입력을 문자열로 바꿨다가 다시 토큰화하면 실제 입력 길이는 목표와 조금 다를 수 있습니다.

In [ ]:
def benchmark(input_tokens, output_tokens, concurrency):
    command = [
        *VLLM, "bench", "serve",
        "--backend", "openai",
        "--base-url", BASE_URL,
        "--model", MODEL,
        "--dataset-name", "random",
        "--random-input-len", str(input_tokens),
        "--random-output-len", str(output_tokens),
        "--random-range-ratio", "0",
        "--num-prompts", str(NUM_PROMPTS),
        "--num-warmups", "8",
        "--max-concurrency", str(concurrency),
        "--request-rate", "inf",
        "--temperature", "0",
        "--ignore-eos",
        "--percentile-metrics", "ttft,tpot,e2el",
        "--metric-percentiles", "50,95",
    ]
    for _ in range(REPEATS):
        result = subprocess.run(
            command, stdout=subprocess.PIPE, text=True, check=True,
        )
        print(
            f"input_tokens={input_tokens}, output_tokens={output_tokens}, "
            f"concurrency={concurrency}"
        )
        print(result.stdout.split("Serving Benchmark Result", 1)[1])

## 5. 동시 요청 수에 따른 차이

입력 512토큰, 출력 128토큰을 유지하고 동시 요청 상한만 **1, 4, 16, 32건**으로 바꿉니다. 함께 계산하는 요청이 많아지면 GPU가 같은 시간에 더 많은 출력을 생성할 수 있습니다. 반면 각 요청이 다음 계산을 기다리는 시간은 늘 수 있습니다.

16건과 32건의 **Output token throughput**과 **P95 E2EL**을 읽습니다. 초당 출력은 얼마나 늘었고, 상대적으로 느린 요청은 얼마나 기다렸나요? [발표의 배치 예](chapter-04.html#batch)

In [ ]:
for concurrency in [1, 4, 16, 32]:
    benchmark(input_tokens=512, output_tokens=128, concurrency=concurrency)

## 6. 입력 길이에 따른 차이

짧은 공지에서 회의 시간을 확인하는 경우와 긴 회의록 전체에서 시간을 찾는 경우를 생각해 봅니다. 답은 한 문장이어도 첫 답을 내기 전에 읽을 양이 다릅니다.

입력을 **128, 512, 2,048토큰**으로 바꿉니다. 동시 요청 상한은 4건, 출력은 128토큰으로 고정합니다. **P50 TTFT와 P95 TTFT**를 읽어 첫 응답을 받기까지 얼마나 걸렸는지 확인합니다. [발표의 입력 처리](chapter-04.html#request)

In [ ]:
for input_tokens in [128, 512, 2048]:
    benchmark(input_tokens=input_tokens, output_tokens=128, concurrency=4)

## 7. 출력 길이에 따른 차이

같은 회의록에서 한 줄 요약을 요청할 때와 상세한 정리를 요청할 때는 답변 길이가 달라집니다. 답변이 길어지면 다음 토큰을 정하는 계산도 더 많이 해야 합니다.

입력 512토큰, 동시 요청 상한 4건을 유지하고 출력을 **32, 128, 512토큰**으로 바꿉니다. 전체 답변을 받는 시간인 **P50 E2EL**과 토큰 하나를 더 받는 데 걸리는 시간인 **P50 TPOT**를 비교합니다. 두 값 중 어느 쪽이 출력 길이에 더 크게 영향을 받았나요?

In [ ]:
for output_tokens in [32, 128, 512]:
    benchmark(input_tokens=512, output_tokens=output_tokens, concurrency=4)

## 8. H100과 B300 결과 해석

| 비교할 두 결과 | 설명할 수 있는 범위 |
| --- | --- |
| H100 한 장과 B300 한 장의 직접 실행 | 같은 모델과 입력 8개를 처리한 시간 |
| H100 네 장과 B300 한 장의 서버 실행 | GPU 기종, GPU 수, GPU 사이의 통신을 모두 포함한 서버 구성 전체의 결과 |
| 같은 장비에서 동시 요청 상한만 다른 결과 | 동시에 처리하는 요청 수가 처리량과 사용자 응답 시간에 미친 변화 |

같은 조건으로 세 번 잰 결과에서 속도 차이가 반복해서 나타나는지 확인합니다. H100 한 장의 성능은 2절에서 직접 잰 값으로 설명합니다. 네 장의 결과를 4로 나누면 GPU끼리 결과를 주고받는 시간까지 포함됩니다.

실제 측정값으로 다음 문장을 완성해 보세요. “동시에 ___건을 보냈을 때 초당 출력은 ___토큰이었고, 요청의 약 95%는 ___ms 안에 끝났습니다.” 두 값은 같은 측정에서 읽습니다.

## 9. 서버 종료

서버를 실행한 터미널에서 **Ctrl+C**를 누릅니다. 나중에 셀 출력만 다시 볼 때는 서버를 실행할 필요가 없습니다.

[H100에서 Gemma 4와 DFlash 비교하기](vllm-gemma4-dflash.ipynb)에서는 같은 회의 공지를 사용하되, 입력과 출력 길이를 고정하지 않고 실제 답변과 생성 속도를 비교합니다.

## 참고 자료

- [Inference Korea 5주차](https://songys.github.io/inference-korea/)
- [Inference Engineering 4장 한국어 참고 자료](https://corazzon.github.io/inference-engineering/ko/04-소프트웨어.html)
- [vLLM 서버 실행](https://docs.vllm.ai/en/stable/cli/serve/)
- [vLLM 서버 벤치마크](https://docs.vllm.ai/en/stable/cli/bench/serve/)
- [vLLM 병렬화](https://docs.vllm.ai/en/stable/serving/parallelism_scaling/)
- [Qwen3.8-27B 모델 카드](https://huggingface.co/Qwen/Qwen3.8-27B)
- [Qwen3.8-27B 모델 설정](https://huggingface.co/Qwen/Qwen3.8-27B/blob/main/config.json)
- [Qwen3.8-27B vLLM 실행 안내](https://recipes.vllm.ai/Qwen/Qwen3.8-27B)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)

설명과 실습 코드는 별도로 작성했다. 모델 라이선스는 Apache-2.0이다. 이 노트북에는 교재 본문이나 그림을 복제하지 않았다.